# 02. Model Training

UCI Hydraulic System 데이터를 이용하여
냉각기, 밸브, 펌프, 축압기 상태와 stable_flag 예측 모델을 학습한다.

### 목표
1. 평균 특징 17개 불러오기
2. 축압기 기준 Stratified 70/15/15 분할 불러오기
3. 네 부품과 stable_flag가 동일한 cycle_id 사용
4. RandomForest 기준 모델 확인
5. 10초 / 20초 / 30초 / 60초 비교
6. RandomForest / LightGBM 비교
7. 최종 Test는 03에서만 사용

## 1. 라이브러리

In [1]:
import json
import pandas as pd

from pathlib import Path

from sklearn.ensemble import (
    RandomForestClassifier
)

from sklearn.model_selection import (
    train_test_split
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix
)

from lightgbm import (
    LGBMClassifier
)

## 2. 특징 데이터 준비

### 2-1. 데이터 경로

In [2]:
processed_dir = Path(
    "../data/processed"
)

profile_path = Path(
    "../data/raw/uci_hydraulic/extracted/profile.txt"
)

### 2-2. 특징 파일 존재 여부 확인

In [3]:
feature_files = [
    "features_10s.parquet",
    "features_20s.parquet",
    "features_30s.parquet",
    "features_60s.parquet"
]

for file_name in feature_files:
    file_path = (
        processed_dir
        / file_name
    )

    print(
        file_name,
        "→",
        file_path.exists()
    )

features_10s.parquet → True
features_20s.parquet → True
features_30s.parquet → True
features_60s.parquet → True


### 2-3. 특징 데이터 불러오기

In [4]:
features_10s = pd.read_parquet(
    processed_dir
    / "features_10s.parquet"
)

features_20s = pd.read_parquet(
    processed_dir
    / "features_20s.parquet"
)

features_30s = pd.read_parquet(
    processed_dir
    / "features_30s.parquet"
)

features_60s = pd.read_parquet(
    processed_dir
    / "features_60s.parquet"
)

### 2-4. profile 불러오기

In [5]:
profile = pd.read_csv(
    profile_path,
    sep=r"\s+",
    header=None
)

profile.columns = [
    "cooler",
    "valve",
    "pump",
    "accumulator",
    "stable_flag"
]

profile.insert(
    0,
    "cycle_id",
    range(1, len(profile) + 1)
)

print(
    "profile:",
    profile.shape
)

profile: (2205, 6)


### 2-5. 센서와 최종 특징 목록

In [6]:
sensor_names = [
    "PS1", "PS2", "PS3", "PS4", "PS5", "PS6",
    "EPS1",
    "FS1", "FS2",
    "TS1", "TS2", "TS3", "TS4",
    "VS1",
    "CE", "CP", "SE"
]

sampling_rates = {
    "PS1": 100, "PS2": 100, "PS3": 100,
    "PS4": 100, "PS5": 100, "PS6": 100,
    "EPS1": 100,
    "FS1": 10, "FS2": 10,
    "TS1": 1, "TS2": 1, "TS3": 1, "TS4": 1,
    "VS1": 1,
    "CE": 1, "CP": 1, "SE": 1
}

mean_feature_cols = [
    f"{sensor}_mean"
    for sensor in sensor_names
]

component_order = [
    "cooler",
    "valve",
    "pump",
    "accumulator"
]

# component_order는 JSON의 components 안에 들어갈 4개 부품만 의미한다.
# stable_flag는 별도의 예측 타깃이므로 target_order에 추가한다.
target_order = component_order + ["stable_flag"]

### 2-6. 전체 컬럼 확인

In [7]:
print("전체 컬럼 수 :", len(features_60s.columns))

for col in features_60s.columns:
    print(col)

전체 컬럼 수 : 18
cycle_id
PS1_mean
PS2_mean
PS3_mean
PS4_mean
PS5_mean
PS6_mean
EPS1_mean
FS1_mean
FS2_mean
TS1_mean
TS2_mean
TS3_mean
TS4_mean
VS1_mean
CE_mean
CP_mean
SE_mean


### 2-7. 60초 특징 결측값 확인

In [8]:
missing_values = (
    features_60s
    .isnull()
    .sum()
)

print("전체 결측값 수 :", missing_values.sum())
print("결측값이 있는 컬럼 :")
print(
    missing_values[
        missing_values > 0
    ]
)

전체 결측값 수 : 0
결측값이 있는 컬럼 :
Series([], dtype: int64)


## 3. 평균 특징 17개 검사

### 3-1. 시간별 특징 묶기

In [9]:
feature_frames = {
    "10s": features_10s,
    "20s": features_20s,
    "30s": features_30s,
    "60s": features_60s
}

expected_columns = (
    ["cycle_id"]
    + mean_feature_cols
)

### 3-2. 컬럼 구조 확인

In [10]:
for name, features in (
    feature_frames.items()
):
    if (
        features.columns.tolist()
        != expected_columns
    ):
        raise ValueError(
            f"{name} 특징 컬럼 오류"
        )

    print(
        name,
        "→",
        features.shape
    )

10s → (2205, 18)
20s → (2205, 18)
30s → (2205, 18)
60s → (2205, 18)


### 3-3. 중복·결측값 검사

In [11]:
for name, features in (
    feature_frames.items()
):
    if (
        features["cycle_id"]
        .duplicated()
        .any()
    ):
        raise ValueError(
            f"{name} cycle_id 중복"
        )

    if (
        features[
            mean_feature_cols
        ]
        .isna()
        .any()
        .any()
    ):
        raise ValueError(
            f"{name} 결측값 존재"
        )

print(
    "평균 특징 17개 검사 완료"
)

평균 특징 17개 검사 완료


## 4. 축압기 기준 Stratified 분할

### 4-1. 분할 함수

In [12]:
def make_accumulator_stratified_split(
    profile,
    random_state=42
):
    # 전체의 15%를 Test로 분리
    dev_ids, test_ids = train_test_split(
        profile["cycle_id"],
        test_size=0.15,
        random_state=random_state,
        stratify=profile["accumulator"]
    )

    # 남은 85% 중 전체의 15%를 Validation으로 분리
    dev_profile = profile[
        profile["cycle_id"].isin(dev_ids)
    ].copy()

    train_ids, val_ids = train_test_split(
        dev_profile["cycle_id"],
        test_size=0.15 / 0.85,
        random_state=random_state,
        stratify=dev_profile["accumulator"]
    )

    return (
        sorted(map(int, train_ids)),
        sorted(map(int, val_ids)),
        sorted(map(int, test_ids))
    )

### 4-2. 저장된 분할 파일 확인

In [13]:
split_path = (
    processed_dir
    / "split_ids_accumulator_stratified.json"
)

print(
    "분할 파일 존재:",
    split_path.exists()
)

분할 파일 존재: True


### 4-3. 분할 ID 불러오기 또는 재생성

In [14]:
if split_path.exists():
    with open(
        split_path,
        "r",
        encoding="utf-8"
    ) as file:
        split_data = json.load(
            file
        )

    train_ids = list(
        map(
            int,
            split_data[
                "train_ids"
            ]
        )
    )

    val_ids = list(
        map(
            int,
            split_data[
                "val_ids"
            ]
        )
    )

    test_ids = list(
        map(
            int,
            split_data[
                "test_ids"
            ]
        )
    )

    print(
        "저장된 분할 사용"
    )

else:
    train_ids, val_ids, test_ids = (
        make_accumulator_stratified_split(
            profile,
            random_state=42
        )
    )

    print(
        "분할 파일이 없어 동일 규칙으로 재생성"
    )

저장된 분할 사용


### 4-4. 분할 개수 확인

In [15]:
print(
    "Train:",
    len(train_ids)
)

print(
    "Validation:",
    len(val_ids)
)

print(
    "Test:",
    len(test_ids)
)

assert len(train_ids) == 1543
assert len(val_ids) == 331
assert len(test_ids) == 331

Train: 1543
Validation: 331
Test: 331


### 4-5. cycle_id 중복 확인

In [16]:
assert set(
    train_ids
).isdisjoint(
    val_ids
)

assert set(
    train_ids
).isdisjoint(
    test_ids
)

assert set(
    val_ids
).isdisjoint(
    test_ids
)

print(
    "분할 중복 0개"
)

분할 중복 0개


### 4-6. 축압기 클래스 분포

In [17]:
for split_name, ids in [
    ("Train", train_ids),
    ("Validation", val_ids),
    ("Test", test_ids)
]:
    print(
        f"\n[{split_name}]"
    )

    print(
        profile.loc[
            profile[
                "cycle_id"
            ].isin(ids),
            "accumulator"
        ]
        .value_counts()
        .sort_index()
    )


[Train]
accumulator
90     566
100    279
115    279
130    419
Name: count, dtype: int64

[Validation]
accumulator
90     121
100     60
115     60
130     90
Name: count, dtype: int64

[Test]
accumulator
90     121
100     60
115     60
130     90
Name: count, dtype: int64


### 4-7. stable_flag 라벨 분포 확인

`stable_flag`는 모델 입력 특징(X)에 넣지 않고,
냉각기·밸브·펌프·축압기와 동일하게 **별도의 y 예측 대상**으로 사용한다.
메인 분할은 그대로 축압기 기준 Stratified 70/15/15를 사용한다.

In [18]:
for split_name, ids in [
    ("Train", train_ids),
    ("Validation", val_ids),
    ("Test", test_ids)
]:
    stable_counts = (
        profile.loc[
            profile["cycle_id"].isin(ids),
            "stable_flag"
        ]
        .value_counts()
        .sort_index()
    )

    print(f"\n[{split_name}] stable_flag")
    print(stable_counts)

    # 각 분할에 0과 1이 모두 존재하는지 확인
    if set(stable_counts.index) != {0, 1}:
        raise ValueError(
            f"{split_name}에 stable_flag 0/1이 모두 존재하지 않습니다."
        )


[Train] stable_flag
stable_flag
0    1005
1     538
Name: count, dtype: int64

[Validation] stable_flag
stable_flag
0    216
1    115
Name: count, dtype: int64

[Test] stable_flag
stable_flag
0    228
1    103
Name: count, dtype: int64


## 5. 학습용 X / y 생성

### 5-1. 공통 함수

In [19]:
def get_xy(
    features,
    ids,
    component
):
    feature_part = (
        features[
            features["cycle_id"].isin(ids)
        ]
        .sort_values("cycle_id")
        .reset_index(drop=True)
    )

    label_part = (
        profile[
            profile["cycle_id"].isin(ids)
        ]
        .sort_values("cycle_id")
        .reset_index(drop=True)
    )

    if (
        feature_part["cycle_id"].tolist()
        != label_part["cycle_id"].tolist()
    ):
        raise ValueError(
            "특징 데이터와 라벨의 cycle_id 순서가 다릅니다."
        )

    X = feature_part[mean_feature_cols].copy()
    y = label_part[component].copy()

    return X, y

### 5-2. 함수 동작 확인

In [20]:
X_train_check, y_train_check = (
    get_xy(
        features_20s,
        train_ids,
        "accumulator"
    )
)

print(
    "X_train:",
    X_train_check.shape
)

print(
    "y_train:",
    y_train_check.shape
)

assert (
    X_train_check.shape[1]
    == 17
)

X_train: (1543, 17)
y_train: (1543,)


## 6. 60초 RandomForest 기준 모델

### 6-1. 모델 학습

In [21]:
rf_60s_models = {}
rf_60s_results = []

for component in target_order:
    X_train, y_train = get_xy(
        features_60s,
        train_ids,
        component
    )

    X_val, y_val = get_xy(
        features_60s,
        val_ids,
        component
    )

    model = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train
    )

    pred = model.predict(
        X_val
    )

    rf_60s_models[
        component
    ] = model

    rf_60s_results.append({
        "component":
            component,
        "accuracy":
            accuracy_score(
                y_val,
                pred
            ),
        "macro_f1":
            f1_score(
                y_val,
                pred,
                average="macro",
                zero_division=0
            )
    })

### 6-2. 60초 기준 모델 결과

In [22]:
rf_60s_results = pd.DataFrame(
    rf_60s_results
)

display(
    rf_60s_results
)

,component,accuracy,macro_f1
0,cooler,1.000000,1.000000
1,valve,0.966767,0.956952
2,pump,0.996979,0.994926
3,accumulator,0.969789,0.962960
4,stable_flag,0.975831,0.973349


### 6-3. 60초 Validation 혼동행렬

In [23]:
for component in target_order:
    X_val, y_val = get_xy(
        features_60s,
        val_ids,
        component
    )

    pred = (
        rf_60s_models[component]
        .predict(X_val)
    )

    labels = sorted(
        profile[component].unique()
    )

    cm = confusion_matrix(
        y_val,
        pred,
        labels=labels
    )

    print("=" * 50)
    print(component)
    print("Labels :", labels)
    print(cm)

cooler
Labels : [np.int64(3), np.int64(20), np.int64(100)]
[[115   0   0]
 [  0 104   0]
 [  0   0 112]]
valve
Labels : [np.int64(73), np.int64(80), np.int64(90), np.int64(100)]
[[ 47   2   0   0]
 [  0  54   0   0]
 [  1   1  43   4]
 [  0   0   3 176]]
pump
Labels : [np.int64(0), np.int64(1), np.int64(2)]
[[199   0   0]
 [  0  61   1]
 [  0   0  70]]
accumulator
Labels : [np.int64(90), np.int64(100), np.int64(115), np.int64(130)]
[[118   3   0   0]
 [  1  57   2   0]
 [  0   4  56   0]
 [  0   0   0  90]]
stable_flag
Labels : [np.int64(0), np.int64(1)]
[[212   4]
 [  4 111]]


## 7. 10초 / 20초 / 30초 / 60초 비교

### 7-1. 시간별 특징 준비

In [24]:
window_features = {
    10: features_10s,
    20: features_20s,
    30: features_30s,
    60: features_60s
}

### 7-2. RandomForest 시간별 학습

In [25]:
rf_window_models = {}
rf_window_results = []

for seconds, features in (
    window_features.items()
):
    rf_window_models[
        seconds
    ] = {}

    for component in target_order:
        X_train, y_train = get_xy(
            features,
            train_ids,
            component
        )

        X_val, y_val = get_xy(
            features,
            val_ids,
            component
        )

        model = RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            X_train,
            y_train
        )

        pred = model.predict(
            X_val
        )

        rf_window_models[
            seconds
        ][
            component
        ] = model

        rf_window_results.append({
            "window_sec":
                seconds,
            "component":
                component,
            "accuracy":
                accuracy_score(
                    y_val,
                    pred
                ),
            "macro_f1":
                f1_score(
                    y_val,
                    pred,
                    average="macro",
                    zero_division=0
                )
        })

### 7-3. 시간별 부품 성능

In [26]:
rf_window_results = pd.DataFrame(
    rf_window_results
)

display(
    rf_window_results
)

,window_sec,component,accuracy,macro_f1
0,10,cooler,1.000000,1.000000
1,10,valve,1.000000,1.000000
2,10,pump,0.987915,0.982810
3,10,accumulator,0.984894,0.982582
4,10,stable_flag,0.981873,0.980012
5,20,cooler,1.000000,1.000000
6,20,valve,0.993958,0.991672
7,20,pump,0.993958,0.991390
8,20,accumulator,0.969789,0.961990
9,20,stable_flag,0.963746,0.960024


### 7-4. 시간별 평균 성능

In [27]:
rf_window_summary = (
    rf_window_results
    .groupby(
        "window_sec"
    )[
        [
            "accuracy",
            "macro_f1"
        ]
    ]
    .mean()
)

display(
    rf_window_summary
)

,accuracy,macro_f1
window_sec,,
10,0.990937,0.989081
20,0.984290,0.981015
30,0.984894,0.982383
60,0.981873,0.977638


### 7-5. 축압기 10초 혼동행렬 확인

In [28]:
X_val_acc_10s, y_val_acc_10s = get_xy(
    features_10s,
    val_ids,
    "accumulator"
)

pred_acc_10s = (
    rf_window_models[10]["accumulator"]
    .predict(X_val_acc_10s)
)

acc_labels = sorted(
    profile["accumulator"].unique()
)

cm_acc_10s = confusion_matrix(
    y_val_acc_10s,
    pred_acc_10s,
    labels=acc_labels
)

print("축압기 라벨 :", acc_labels)
print(cm_acc_10s)

축압기 라벨 : [np.int64(90), np.int64(100), np.int64(115), np.int64(130)]
[[120   1   0   0]
 [  1  58   1   0]
 [  0   0  58   2]
 [  0   0   0  90]]


## 8. RandomForest / LightGBM 비교

### 8-1. 20초 비교 모델 준비

In [29]:
compare_results = []
rf_20s_models = {}
lgb_20s_models = {}

### 8-2. 20초 모델 학습 및 평가

In [30]:
for component in target_order:
    X_train, y_train = get_xy(
        features_20s,
        train_ids,
        component
    )

    X_val, y_val = get_xy(
        features_20s,
        val_ids,
        component
    )

    rf_model = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )

    lgb_model = LGBMClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    for model_name, model in [
        (
            "RandomForest",
            rf_model
        ),
        (
            "LightGBM",
            lgb_model
        )
    ]:
        model.fit(
            X_train,
            y_train
        )

        pred = model.predict(
            X_val
        )

        compare_results.append({
            "component":
                component,
            "model":
                model_name,
            "accuracy":
                accuracy_score(
                    y_val,
                    pred
                ),
            "macro_f1":
                f1_score(
                    y_val,
                    pred,
                    average="macro",
                    zero_division=0
                )
        })

    rf_20s_models[
        component
    ] = rf_model

    lgb_20s_models[
        component
    ] = lgb_model

### 8-3. 부품별 모델 비교

In [31]:
compare_results = pd.DataFrame(
    compare_results
)

display(
    compare_results
)

,component,model,accuracy,macro_f1
0,cooler,RandomForest,1.000000,1.000000
1,cooler,LightGBM,1.000000,1.000000
2,valve,RandomForest,0.993958,0.991672
3,valve,LightGBM,0.996979,0.995138
4,pump,RandomForest,0.993958,0.991390
5,pump,LightGBM,0.993958,0.993225
6,accumulator,RandomForest,0.969789,0.961990
7,accumulator,LightGBM,0.969789,0.965018
8,stable_flag,RandomForest,0.963746,0.960024
9,stable_flag,LightGBM,0.978852,0.976820


### 8-4. 모델별 평균 성능

In [32]:
compare_summary = (
    compare_results
    .groupby(
        "model"
    )[
        [
            "accuracy",
            "macro_f1"
        ]
    ]
    .mean()
    .sort_values(
        "macro_f1",
        ascending=False
    )
)

display(
    compare_summary
)

,accuracy,macro_f1
model,,
LightGBM,0.987915,0.986040
RandomForest,0.984290,0.981015


### 8-5. 부품별 우수 모델 확인

In [33]:
model_f1_table = (
    compare_results
    .pivot(
        index="component",
        columns="model",
        values="macro_f1"
    )
)

model_f1_table = model_f1_table.loc[
    target_order,
    ["RandomForest", "LightGBM"]
]

display(model_f1_table)

model,RandomForest,LightGBM
component,,
cooler,1.000000,1.000000
valve,0.991672,0.995138
pump,0.991390,0.993225
accumulator,0.961990,0.965018
stable_flag,0.960024,0.976820


In [ ]:
best_model_results = []

for component in target_order:
    rf_score = model_f1_table.loc[
        component,
        "RandomForest"
    ]

    lgb_score = model_f1_table.loc[
        component,
        "LightGBM"
    ]

    if rf_score > lgb_score:
        best_model = "RandomForest"
        best_score = rf_score
    elif lgb_score > rf_score:
        best_model = "LightGBM"
        best_score = lgb_score
    else:
        best_model = "Same"
        best_score = rf_score

    best_model_results.append({
        "component": component,
        "best_model": best_model,
        "best_macro_f1": best_score
    })

best_model_results = pd.DataFrame(
    best_model_results
)

display(best_model_results)

## 8-6. 성능 개선 및 검증 방식 비교

기존 노트북의 `class_weight`, Stratified 비교, 시간순 Rolling 실험을 유지한다.
단, **메인 학습 분할은 이미 축압기 기준 Stratified 70/15/15로 확정**되어 있으므로
아래 시간순 실험은 강건성 확인용이다.

### 8-6-1. class_weight='balanced' RandomForest 비교

In [ ]:
balanced_results = []

for component in target_order:
    X_train, y_train = get_xy(
        features_60s,
        train_ids,
        component
    )
    X_val, y_val = get_xy(
        features_60s,
        val_ids,
        component
    )

    balanced_model = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    )

    balanced_model.fit(
        X_train,
        y_train
    )

    pred = balanced_model.predict(
        X_val
    )

    balanced_results.append({
        "component": component,
        "accuracy": accuracy_score(
            y_val,
            pred
        ),
        "macro_f1": f1_score(
            y_val,
            pred,
            average="macro",
            zero_division=0
        )
    })

balanced_results = pd.DataFrame(
    balanced_results
)

display(balanced_results)

#### class_weight 비교 결과

이 셀은 클래스 불균형이 성능 저하의 주원인인지 확인하기 위한 실험이다.
실제 결과는 노트북을 다시 실행한 뒤 갱신한다.

### 8-6-2. 메인 Stratified와 시간순서 분할 비교

In [ ]:
# 메인 Stratified 성능: 60초 RandomForest Validation 결과
stratified_compare = (
    rf_60s_results[
        [
            "component",
            "accuracy",
            "macro_f1"
        ]
    ]
    .rename(
        columns={
            "accuracy":
                "stratified_accuracy",
            "macro_f1":
                "stratified_macro_f1"
        }
    )
)

# 시간순서 분할은 강건성 비교용으로만 사용
time_train_ids = list(
    range(1, 1544)
)
time_val_ids = list(
    range(1544, 1875)
)

time_results = []

for component in target_order:
    X_time_train, y_time_train = get_xy(
        features_60s,
        time_train_ids,
        component
    )
    X_time_val, y_time_val = get_xy(
        features_60s,
        time_val_ids,
        component
    )

    time_model = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )

    time_model.fit(
        X_time_train,
        y_time_train
    )

    pred = time_model.predict(
        X_time_val
    )

    time_results.append({
        "component": component,
        "time_accuracy": accuracy_score(
            y_time_val,
            pred
        ),
        "time_macro_f1": f1_score(
            y_time_val,
            pred,
            labels=sorted(
                profile[component].unique()
            ),
            average="macro",
            zero_division=0
        )
    })

time_results = pd.DataFrame(
    time_results
)

split_compare = (
    time_results
    .merge(
        stratified_compare,
        on="component"
    )
)

display(split_compare)

#### 분할 방식 해석

메인 성능 평가는 축압기 클래스 비율을 보존한 Stratified 분할을 사용한다.
시간순서 결과는 뒤쪽 구간의 분포 변화에 대한 추가 강건성 검증으로 해석한다.

### 8-6-3. 시간순 Rolling 추가 검증

In [ ]:
rolling_folds = [
    {
        "fold": "1차",
        "train_start": 1,
        "train_end": 1000,
        "val_start": 1001,
        "val_end": 1200
    },
    {
        "fold": "2차",
        "train_start": 1,
        "train_end": 1200,
        "val_start": 1201,
        "val_end": 1400
    },
    {
        "fold": "3차",
        "train_start": 1,
        "train_end": 1400,
        "val_start": 1401,
        "val_end": 1600
    },
    {
        "fold": "4차",
        "train_start": 1,
        "train_end": 1600,
        "val_start": 1601,
        "val_end": 1800
    }
]

rolling_results = []

for fold in rolling_folds:
    fold_train_ids = list(
        range(
            fold["train_start"],
            fold["train_end"] + 1
        )
    )
    fold_val_ids = list(
        range(
            fold["val_start"],
            fold["val_end"] + 1
        )
    )

    for component in target_order:
        X_fold_train, y_fold_train = get_xy(
            features_60s,
            fold_train_ids,
            component
        )
        X_fold_val, y_fold_val = get_xy(
            features_60s,
            fold_val_ids,
            component
        )

        model = RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            X_fold_train,
            y_fold_train
        )

        pred = model.predict(
            X_fold_val
        )

        rolling_results.append({
            "fold": fold["fold"],
            "component": component,
            "train_range":
                f'{fold["train_start"]}~{fold["train_end"]}',
            "val_range":
                f'{fold["val_start"]}~{fold["val_end"]}',
            "accuracy":
                accuracy_score(
                    y_fold_val,
                    pred
                ),
            "macro_f1":
                f1_score(
                    y_fold_val,
                    pred,
                    labels=sorted(
                        profile[component].unique()
                    ),
                    average="macro",
                    zero_division=0
                )
        })

rolling_results = pd.DataFrame(
    rolling_results
)

display(rolling_results)

### 8-6-4. 시간별 특징에 동일 공통 Stratified 분할 적용

In [ ]:
# 부품마다 새 분할을 만들지 않는다.
# 축압기 기준으로 만든 공통 train_ids / val_ids를
# 10/20/30/60초에 동일하게 적용한다.
common_split_window_results = []

feature_sets = {
    "10s": features_10s,
    "20s": features_20s,
    "30s": features_30s,
    "60s": features_60s
}

for window, features in (
    feature_sets.items()
):
    for component in target_order:
        X_train, y_train = get_xy(
            features,
            train_ids,
            component
        )
        X_val, y_val = get_xy(
            features,
            val_ids,
            component
        )

        model = RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            X_train,
            y_train
        )

        pred = model.predict(
            X_val
        )

        common_split_window_results.append({
            "window": window,
            "component": component,
            "accuracy":
                accuracy_score(
                    y_val,
                    pred
                ),
            "macro_f1":
                f1_score(
                    y_val,
                    pred,
                    average="macro",
                    zero_division=0
                )
        })

common_split_window_results = pd.DataFrame(
    common_split_window_results
)

display(
    common_split_window_results
)

## 9. Validation 혼동행렬

### 9-1. LightGBM 20초 혼동행렬

In [ ]:
for component in target_order:
    X_val, y_val = get_xy(
        features_20s,
        val_ids,
        component
    )

    model = (
        lgb_20s_models[
            component
        ]
    )

    pred = model.predict(
        X_val
    )

    labels = sorted(
        profile[
            component
        ].unique()
    )

    cm = confusion_matrix(
        y_val,
        pred,
        labels=labels
    )

    cm_df = pd.DataFrame(
        cm,
        index=[
            f"actual_{label}"
            for label in labels
        ],
        columns=[
            f"pred_{label}"
            for label in labels
        ]
    )

    print(
        f"\n===== {component.upper()} ====="
    )

    display(
        cm_df
    )

## 10. 학습 단계 최종 확인

### 10-1. 특징 수 확인

In [ ]:
print(
    "입력 특징 수:",
    len(mean_feature_cols)
)

print(
    "입력 특징:",
    mean_feature_cols
)

assert len(
    mean_feature_cols
) == 17

print("예측 타깃 :", target_order)

### 10-2. Test 미사용 확인

In [ ]:
assert set(
    train_ids
).isdisjoint(
    test_ids
)

assert set(
    val_ids
).isdisjoint(
    test_ids
)

print(
    "02에서는 Validation까지만 사용"
)

print(
    "최종 Test 평가는 03에서 수행"
)